# Module 04-2: 이메일 분류 에이전트 설계 및 구현 (솔루션)

## 학습 목표

1. **State 직접 설계**: 이메일 분류에 필요한 `EmailClassifierState`를 스스로 설계한다
2. **4노드 구현**: parse_email → classify_email → assess_urgency → generate_action 흐름을 구현한다
3. **조건부 분기**: 스팸 이메일은 `assess_urgency`를 건너뛰는 분기를 구현한다
4. **3가지 테스트**: 정상 업무 이메일, 스팸, 빈 이메일을 모두 테스트한다

---
> 참고 문서: `docs/part2-first-agent/04-building-first-agent.md`

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import json
from typing import TypedDict
from functools import partial
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: EmailClassifierState 설계 (솔루션)

In [ ]:
# 솔루션
class EmailClassifierState(TypedDict):
    raw_email: str
    sender: str | None
    subject: str | None
    body: str | None
    category: str | None   # "spam" 또는 "ham"
    urgency: str | None    # "high", "medium", "low"
    action: str | None
    current_step: str
    error: str | None

print(f"설계된 필드: {list(EmailClassifierState.__annotations__.keys())}")

In [ ]:
# 검증 셀
assert 'EmailClassifierState' in dir()
fields = list(EmailClassifierState.__annotations__.keys())
required = ['raw_email', 'category', 'urgency', 'action', 'current_step', 'error']
for f in required:
    assert f in fields, f"'{f}' 필드가 필요합니다"
print(f"설계된 필드: {fields}")
print("✅ 실습 1 완료!")

## 실습 2: 4노드 구현 (솔루션)

In [ ]:
# 솔루션: parse_email
def parse_email(state: EmailClassifierState) -> dict:
    raw = state.get("raw_email", "")
    if not raw or not raw.strip():
        return {"error": "이메일이 비어있습니다.", "current_step": "error"}
    lines = raw.strip().split("\n")
    sender = ""
    subject = ""
    body_lines = []
    in_body = False
    for line in lines:
        if line.startswith("From:"):
            sender = line[5:].strip()
        elif line.startswith("Subject:"):
            subject = line[8:].strip()
        else:
            in_body = True
            body_lines.append(line)
    body = "\n".join(body_lines).strip()
    print(f"  [parse_email] 파싱 완료: sender='{sender}', subject='{subject}'")
    return {"sender": sender, "subject": subject, "body": body, "current_step": "parsed"}


# 솔루션: classify_email
def classify_email(state: EmailClassifierState, llm=None) -> dict:
    if llm is None:
        return {"error": "LLM이 설정되지 않았습니다.", "current_step": "error"}
    subject = state.get("subject", "")
    body = state.get("body", "")
    prompt = f"다음 이메일을 분류해주세요. 스팸이면 'spam', 정상이면 'ham'으로 답하세요.\n제목: {subject}\n본문: {body}"
    try:
        response = llm.invoke(prompt)
        content = response.content.strip().lower()
        category = "spam" if "spam" in content else "ham"
        print(f"  [classify_email] 분류 완료: {category}")
        return {"category": category, "current_step": "classified"}
    except Exception as e:
        return {"error": f"분류 중 오류: {str(e)}", "current_step": "error"}


# 솔루션: assess_urgency
def assess_urgency(state: EmailClassifierState, llm=None) -> dict:
    if llm is None:
        return {"error": "LLM이 설정되지 않았습니다.", "current_step": "error"}
    subject = state.get("subject", "")
    body = state.get("body", "")
    prompt = f"다음 이메일의 긴급도를 평가해주세요. high, medium, low 중 하나로 답하세요.\n제목: {subject}\n본문: {body}"
    try:
        response = llm.invoke(prompt)
        content = response.content.strip().lower()
        if "high" in content or "긴급" in content:
            urgency = "high"
        elif "low" in content:
            urgency = "low"
        else:
            urgency = "medium"
        print(f"  [assess_urgency] 긴급도: {urgency}")
        return {"urgency": urgency, "current_step": "assessed"}
    except Exception as e:
        return {"error": f"긴급도 평가 중 오류: {str(e)}", "current_step": "error"}


# 솔루션: generate_action
def generate_action(state: EmailClassifierState) -> dict:
    category = state.get("category", "ham")
    urgency = state.get("urgency", "medium")
    if category == "spam":
        action = "스팸 이메일로 분류됩니다. 스팸 폴더로 이동하고 발신자를 차단하세요."
    elif urgency == "high":
        action = "긴급 이메일입니다. 즉시 확인하고 2시간 이내에 답변하세요."
    elif urgency == "medium":
        action = "일반 업무 이메일입니다. 오늘 중으로 확인하고 처리하세요."
    else:
        action = "낮은 우선순위 이메일입니다. 시간 나는 대로 확인하세요."
    print(f"  [generate_action] 액션 생성 완료")
    return {"action": action, "current_step": "completed"}


print("4개 노드 구현 완료")

In [ ]:
# 검증 셀
base_state = {
    "raw_email": "", "sender": None, "subject": None, "body": None,
    "category": None, "urgency": None, "action": None,
    "current_step": "start", "error": None
}
sample_email = "From: alice@example.com\nSubject: 미팅 일정 확인\n\n내일 오후 2시에 미팅이 있습니다."
r_parse = parse_email({**base_state, "raw_email": sample_email})
assert r_parse.get("sender") is not None
assert r_parse.get("subject") is not None

r_empty = parse_email({**base_state, "raw_email": ""})
assert r_empty.get("error") is not None
print("✅ parse_email 검증 완료!")

## 실습 3: 조건부 분기 및 그래프 조립 (솔루션)

In [ ]:
# 솔루션
def _route_after_parse(state: EmailClassifierState) -> str:
    return "error" if state.get("error") else "classify"


def _route_after_classify(state: EmailClassifierState) -> str:
    return "action" if state.get("category") == "spam" else "urgency"


def build_email_classifier_graph(llm=None):
    graph = StateGraph(EmailClassifierState)
    graph.add_node("parse_email", parse_email)
    graph.add_node("classify_email", partial(classify_email, llm=llm))
    graph.add_node("assess_urgency", partial(assess_urgency, llm=llm))
    graph.add_node("generate_action", generate_action)
    graph.set_entry_point("parse_email")
    graph.add_conditional_edges(
        "parse_email", _route_after_parse,
        {"classify": "classify_email", "error": END}
    )
    graph.add_conditional_edges(
        "classify_email", _route_after_classify,
        {"urgency": "assess_urgency", "action": "generate_action"}
    )
    graph.add_edge("assess_urgency", "generate_action")
    graph.add_edge("generate_action", END)
    return graph.compile()


print("그래프 조립 함수 정의 완료")

In [ ]:
# 검증 셀
s_ok = {"raw_email": "x", "sender": None, "subject": None, "body": None,
        "category": None, "urgency": None, "action": None, "current_step": "parsed", "error": None}
s_err = {**s_ok, "error": "에러"}
s_spam = {**s_ok, "category": "spam"}
s_ham = {**s_ok, "category": "ham"}

assert _route_after_parse(s_ok) == "classify"
assert _route_after_parse(s_err) == "error"
assert _route_after_classify(s_spam) == "action"
assert _route_after_classify(s_ham) == "urgency"
print("✅ 분기 함수 검증 완료!")

## 실습 4: 3가지 테스트 시나리오 (솔루션)

In [ ]:
# 솔루션
email_llm = FakeLLM(
    responses={
        "스팸": "spam",
        "무료": "spam",
        "긴급": "high",
        "미팅": "ham",
    },
    default_response="ham",
)

graph = build_email_classifier_graph(llm=email_llm)

base = {"raw_email": "", "sender": None, "subject": None, "body": None,
        "category": None, "urgency": None, "action": None, "current_step": "start", "error": None}

# 테스트 1: 정상 업무 이메일
result1 = graph.invoke({**base, "raw_email": "From: alice@company.com\nSubject: 내일 미팅 일정 확인\n\n안녕하세요. 내일 오후 2시 미팅 참석 가능하신지 확인 부탁드립니다."})

# 테스트 2: 스팸 이메일
result2 = graph.invoke({**base, "raw_email": "From: promo@spam.com\nSubject: 무료 상품 당첨 축하합니다!\n\n스팸 광고입니다."})

# 테스트 3: 빈 이메일
result3 = graph.invoke({**base, "raw_email": ""})

print(f"테스트 1 (정상): category={result1.get('category')}, action={result1.get('action', '')[:40]}")
print(f"테스트 2 (스팸): category={result2.get('category')}, urgency={result2.get('urgency')}")
print(f"테스트 3 (빈 이메일): error={result3.get('error')}")

In [ ]:
# E2E 검증 셀
assert result1.get("error") is None
assert result1.get("action") is not None
assert result1.get("current_step") == "completed"

assert result2.get("error") is None
assert result2.get("action") is not None
if result2.get("category") == "spam":
    assert result2.get("urgency") is None, "스팸 이메일에서는 urgency가 평가되지 않아야 합니다"

assert result3.get("error") is not None
assert result3.get("current_step") == "error"

print("✅ 실습 4 완료! 3가지 테스트 시나리오 모두 통과")

## 정리

### 오늘 배운 내용
- **State 직접 설계**: 워크플로우에 맞는 State 필드를 스스로 결정하는 능력
- **스팸 건너뛰기 분기**: `_route_after_classify`로 스팸 이메일의 긴급도 평가를 건너뜀
- **3가지 테스트 패턴**: 정상/스팸/에러 케이스를 모두 테스트하는 습관
- **의존성 주입**: `functools.partial`로 LLM을 외부에서 주입하는 패턴 반복 연습